To Retrieve Dataset from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

To Import Required Libraries

In [ ]:
import os
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

Labelling and Collection of data in different classes

In [ ]:
#Data Collection from drive of folder "comp_bio_database_pulses"
data_dir = '/content/drive/MyDrive/comp_bio_database_pulses'
IMG_SIZE = 64
valid_exts = ['.jpg', '.jpeg', '.png']  #valid file extensions

class_names = sorted([   name for name in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, name))
])
num_classes = len(class_names)
#labels and class
X, y = [], []
for label, class_name in enumerate(class_names):
    class_folder = os.path.join(data_dir, class_name)
    for img_file in tqdm(os.listdir(class_folder), desc=f"Loading {class_name}"):
        ext = os.path.splitext(img_file)[-1].lower()
        if ext in valid_exts:
            img_path = os.path.join(class_folder, img_file)
            try:
                img = cv2.imread(img_path)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                X.append(img)
                y.append(label)
            except:
                print(f"Could not read: {img_path}")
        else:
            print(f"Unsupported file: {img_file}")

X = np.array(X).astype('float32') / 255.0
y = to_categorical(np.array(y), num_classes=num_classes)

Split dataset for training and testing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Building CNN Model

In [ ]:
#Build CNN Model
model = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2,2),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

To check data loaded correctly

In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

To Checck whether it has worked on GPU

In [ ]:
import tensorflow as tf
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("TensorFlow is using:", tf.__version__)

In [ ]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

CNN Model Training using 10 Epoches

In [ ]:
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))

Saving the trained model which ran on Colab GPU

In [ ]:
model.save('/content/drive/MyDrive/trained_pulse_model_retrain.h5')
print("Model saved successfully!")

Crosschecking the CNN Test and Train Accuracies

In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"CNN Train Accuracy: {train_acc * 100:.2f}%")
print(f"CNN Test Accuracy: {test_acc * 100:.2f}%")

Loading Saved Training Model for Feature Extraction

In [ ]:
from tensorflow.keras.models import load_model
model = load_model('/content/drive/MyDrive/trained_pulse_model_retrain.h5')

Feature Extraction from the Trained CNN Model which will be used for KNN for classification

In [ ]:
from tensorflow.keras.models import Model # Import Model

feature_extractor = Model(inputs=model.layers[0].input, outputs=model.get_layer(index=-2).output)

X_train_feat = feature_extractor.predict(X_train)
X_test_feat = feature_extractor.predict(X_test)

y_train_labels = y_train.argmax(axis=1)
y_test_labels = y_test.argmax(axis=1)

Training of KNN, Prediction and accuracy checked

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_feat, y_train_labels)

y_pred = knn.predict(X_test_feat)

acc = accuracy_score(y_test_labels, y_pred)
print(f"KNN Accuracy with trained CNN features: {acc * 100:.2f}%")

Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test_labels, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("KNN Confusion Matrix")
plt.show()

Classification Report

In [ ]:
print("\nClassification Report:\n")
print(classification_report(y_test_labels, y_pred, target_names=class_names))

Prediction of Test Sample-1

In [ ]:
test_img_path = '/content/drive/MyDrive/comp_bio_database_pulses/test.jpg'

img = cv2.imread(test_img_path)

if img is None:
    print(f"Couldn't load image at: {test_img_path}")
else:
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img_input = img_resized.astype('float32') / 255.0
    img_input = np.expand_dims(img_input, axis=0)

    test_feat = feature_extractor.predict(img_input)
    pred_label = knn.predict(test_feat)[0]
    pred_class = class_names[pred_label]

    plt.imshow(img_rgb)
    plt.title(f"Predicted: {pred_class}")
    plt.axis('off')
    plt.show()


Prediction of Test Sample-2

In [ ]:
test_img_path = '/content/drive/MyDrive/comp_bio_database_pulses/test2.jpg'

img = cv2.imread(test_img_path)

if img is None:
    print(f"Couldn't load image at: {test_img_path}")
else:
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img_input = img_resized.astype('float32') / 255.0
    img_input = np.expand_dims(img_input, axis=0)

    test_feat = feature_extractor.predict(img_input)
    pred_label = knn.predict(test_feat)[0]
    pred_class = class_names[pred_label]

    plt.imshow(img_rgb)
    plt.title(f"Predicted: {pred_class}")
    plt.axis('off')
    plt.show()


In [ ]:
model.summary()

In [ ]:
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import numpy as np

#Helper function to plot
def plot_2d(X_2d, labels, method, class_names):
    plt.figure(figsize=(10, 6))
    for label in np.unique(labels):
        idx = labels == label
        plt.scatter(X_2d[idx, 0], X_2d[idx, 1], label=class_names[label], alpha=0.7)
    plt.legend()
    plt.title(f"{method} Visualization of CNN Features")
    plt.xlabel("Dim 1")
    plt.ylabel("Dim 2")
    plt.grid(True)
    plt.show()

#PCA
start = time.time()
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_test_feat)
end = time.time()
print(f"PCA done in {end - start:.2f} seconds")
plot_2d(X_pca, y_test_labels, method="PCA", class_names=class_names)

# t-SNE (can be slow)
start = time.time()
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_test_feat)
end = time.time()
print(f"t-SNE done in {end - start:.2f} seconds")
plot_2d(X_tsne, y_test_labels, method="t-SNE", class_names=class_names)